# Notebook 4 — TGN Training (Kaggle)
Upload `training_pairs.pt` and all `graph_seed*.pt` files to a Kaggle dataset called `epidemic-data`.
Run on **Kaggle P100 GPU**.
Key fixes vs previous version:
- Trains on 10 different graphs (not just one)
- Memory reset to noise (not zeros) → prevents identity memorization
- Node feature dropout → prevents over-reliance on specific attributes
- Delta loss + neighbor pressure loss → teaches transmission dynamics
- Per-graph batching → model sees correct graph structure each step

In [ ]:
# Cell 1 — Paths
import os
INPUT = '/kaggle/input/epidemic-data'
BASE  = '/kaggle/working'
os.makedirs(f'{BASE}/model', exist_ok=True)

print('Input files:')
for f in sorted(os.listdir(INPUT)):
    print(' ', f)

In [ ]:
# Cell 2 — Install
import subprocess
subprocess.run([
    'pip','install','-q','torch-geometric','torch-scatter','torch-sparse',
    '-f','https://data.pyg.org/whl/torch-2.2.0+cu118.html'
], check=True)

import torch
print(f'Torch: {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
print(f'GPU:   {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Cell 3 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
import numpy as np
import random
import glob
import json
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

In [ ]:
# Cell 4 — Load all graphs and training pairs
SEEDS = [0, 7, 13, 21, 42, 55, 77, 88, 99, 111]

# Load graph data for each seed
graphs = {}
for seed in SEEDS:
    g = torch.load(f'{INPUT}/graph_seed{seed}.pt', map_location=device)
    graphs[seed] = {
        'edge_index': g.edge_index.to(device),
        'edge_attr':  g.edge_attr.to(device),
        'node_feats': g.x.to(device),
        'N':          g.num_nodes,
    }
    print(f'Graph seed={seed}: N={g.num_nodes}, E={g.edge_index.shape[1]}')

# Load training pairs
all_pairs = torch.load(f'{INPUT}/training_pairs.pt', map_location='cpu')
print(f'\nTotal training pairs: {len(all_pairs)}')

# Show distribution across graphs
from collections import Counter
dist = Counter(p['graph_seed'] for p in all_pairs)
print('Pairs per graph seed:', dict(sorted(dist.items())))

In [ ]:
# Cell 5 — Split by graph seed (not by index) to prevent leakage
# Test seeds: 88, 99, 111 (3 unseen graphs)
# Val seeds:  55, 77       (2 graphs)
# Train seeds: remaining   (5 graphs)

TEST_SEEDS  = {88, 99, 111}
VAL_SEEDS   = {55, 77}
TRAIN_SEEDS = {0, 7, 13, 21, 42}

train_pairs = [p for p in all_pairs if p['graph_seed'] in TRAIN_SEEDS]
val_pairs   = [p for p in all_pairs if p['graph_seed'] in VAL_SEEDS]
test_pairs  = [p for p in all_pairs if p['graph_seed'] in TEST_SEEDS]

print(f'Train: {len(train_pairs)} pairs from seeds {TRAIN_SEEDS}')
print(f'Val:   {len(val_pairs)} pairs from seeds {VAL_SEEDS}')
print(f'Test:  {len(test_pairs)} pairs from seeds {TEST_SEEDS}')
print()
print('NOTE: Test graphs were never seen during training.')
print('      Good test performance = model learned dynamics, not graph identity.')

In [ ]:
# Cell 6 — Model definition
# KEY CHANGES from previous version:
# 1. Memory reset to noise (not zeros) in rollout
# 2. Feature dropout in _step during training
# 3. No fixed N — model takes N as parameter per forward pass

class NodeMemory(nn.Module):
    def __init__(self, mem_dim):
        super().__init__()
        self.mem_dim = mem_dim
        self.memory  = None   # created dynamically per graph

    def init(self, N, device):
        """Called at the start of each rollout with the current graph's N."""
        # Small random noise — prevents model from memorizing node identities
        self.memory = torch.randn(N, self.mem_dim, device=device) * 0.01

    def get(self):       return self.memory
    def update(self, m): self.memory = m.detach()


class EpidemicTGN(nn.Module):
    def __init__(self, node_feat_dim=9, mem_dim=32, state_dim=5):
        super().__init__()
        self.mem_dim   = mem_dim
        self.state_dim = state_dim
        self.memory    = NodeMemory(mem_dim)

        msg_in = mem_dim * 2 + 1 + state_dim
        self.msg_fn = nn.Sequential(
            nn.Linear(msg_in, 64), nn.ReLU(), nn.Linear(64, mem_dim)
        )
        self.mem_updater = nn.GRUCell(mem_dim, mem_dim)

        gat_in = mem_dim + node_feat_dim + state_dim
        self.gat1  = GATv2Conv(gat_in, 32, heads=2, concat=False,
                               edge_dim=1, dropout=0.1, add_self_loops=True)
        self.gat2  = GATv2Conv(32, 32, heads=2, concat=False,
                               edge_dim=1, dropout=0.1, add_self_loops=True)
        self.norm1 = nn.LayerNorm(32)
        self.norm2 = nn.LayerNorm(32)
        self.predictor = nn.Sequential(
            nn.Linear(32, 64), nn.LayerNorm(64), nn.ReLU(),
            nn.Dropout(0.2),   nn.Linear(64, state_dim)
        )

    def _step(self, x_state, node_feats, edge_index, edge_attr):
        x_state    = x_state.float()
        node_feats = node_feats.float()
        edge_attr  = edge_attr.float()
        N   = x_state.shape[0]
        src = edge_index[0]
        dst = edge_index[1]
        mem = self.memory.get().float()

        # --- Feature dropout during training ---
        # Prevents model relying on any single demographic attribute
        if self.training:
            feat_mask  = (torch.rand_like(node_feats) > 0.2).float()
            node_feats = node_feats * feat_mask

        # --- Message passing for memory update ---
        e_w = edge_attr.squeeze(1) if edge_attr.dim() == 2 else edge_attr
        msg_input = torch.cat([
            mem[src], mem[dst], e_w.unsqueeze(1), x_state[src]
        ], dim=-1)
        msgs = self.msg_fn(msg_input).float()

        agg = torch.zeros(N, self.mem_dim, device=mem.device, dtype=torch.float32)
        idx = dst.unsqueeze(1).expand(dst.shape[0], self.mem_dim)
        agg.scatter_add_(0, idx, msgs)

        new_mem = self.mem_updater(agg, mem)
        self.memory.update(new_mem)

        # --- GAT embedding ---
        e_2d = edge_attr if edge_attr.dim() == 2 else edge_attr.unsqueeze(1)
        h = torch.cat([new_mem, node_feats, x_state], dim=-1)
        h = self.norm1(F.elu(self.gat1(h, edge_index, e_2d)))
        h = self.norm2(F.elu(self.gat2(h, edge_index, e_2d)))

        return F.softmax(self.predictor(h), dim=-1)

    def rollout(self, x0, node_feats, edge_index, edge_attr,
                n_days=30, targets=None, tf_prob=1.0, tbptt=10):
        N = x0.shape[0]
        # Init memory with noise for THIS graph's N
        # Critical: noise prevents identity memorization
        self.memory.init(N, x0.device)

        preds = []
        x     = x0.float().detach()

        for t in range(n_days):
            x_next = self._step(x, node_feats, edge_index, edge_attr)
            preds.append(x_next)

            if targets is not None and torch.rand(1).item() < tf_prob:
                x = targets[t].float().detach()
            else:
                x = x_next.detach() if (t + 1) % tbptt == 0 else x_next

        return torch.stack(preds, dim=0)  # (n_days, N, 5)

In [ ]:
# Cell 7 — Loss function with delta loss and neighbor pressure
STATE_WEIGHTS = torch.tensor([0.3, 1.0, 3.0, 1.0, 5.0], device=device)
DAY_WEIGHTS   = torch.linspace(0.5, 2.0, 30, device=device)

def rollout_loss(pred_rollout, target_rollout, x0, edge_index, edge_attr):
    """
    Three-component loss:
    1. Cross-entropy on state predictions
    2. Delta loss — did we predict the RIGHT CHANGE each day?
    3. Neighbor pressure — infected neighbors should drive new exposures

    Components 2 and 3 directly teach transmission dynamics.
    """
    N   = x0.shape[0]
    src = edge_index[0]
    dst = edge_index[1]
    ea  = edge_attr.squeeze(1) if edge_attr.dim() == 2 else edge_attr

    # Prepend x0 for delta computation
    pred_ext   = torch.cat([x0.unsqueeze(0), pred_rollout],   dim=0)  # (31, N, 5)
    target_ext = torch.cat([x0.unsqueeze(0), target_rollout], dim=0)  # (31, N, 5)

    total = torch.tensor(0.0, device=device)

    for t in range(30):
        # 1. Cross-entropy on state
        ce = F.cross_entropy(
            pred_rollout[t],
            target_rollout[t].argmax(dim=-1),
            weight=STATE_WEIGHTS
        )

        # 2. Delta loss on I state — did change match ground truth?
        pred_dI   = pred_ext[t+1, :, 2]   - pred_ext[t, :, 2]
        target_dI = target_ext[t+1, :, 2] - target_ext[t, :, 2]
        delta_loss = F.mse_loss(pred_dI, target_dI)

        # 3. Neighbor pressure — infected neighbors should drive E increase
        # For each node, sum of (neighbor I * edge weight)
        neighbor_I_contrib = pred_rollout[t, src, 2] * ea
        agg_pressure = torch.zeros(N, device=device)
        agg_pressure.scatter_add_(0, dst, neighbor_I_contrib)
        # Ground truth: how much did E actually increase?
        actual_E_gain = (target_ext[t+1, :, 1] - target_ext[t, :, 1]).clamp(0)
        # Normalize both and compare
        pmax = agg_pressure.max() + 1e-8
        emax = actual_E_gain.max() + 1e-8
        pressure_loss = F.mse_loss(agg_pressure / pmax, actual_E_gain / emax)

        total = total + DAY_WEIGHTS[t] * (ce + 2.0 * delta_loss + 0.5 * pressure_loss)

    return total / 30

In [ ]:
# Cell 8 — Teacher forcing schedule
def tf_prob(epoch):
    if epoch < 20: return 1.0
    return max(0.0, 1.0 - (epoch - 20) / 80.0)

print('TF schedule:')
for e in [0, 20, 40, 60, 80, 100]:
    print(f'  Epoch {e:3d}: tf={tf_prob(e):.2f}')

In [ ]:
# Cell 9 — Initialise model
# N is NOT fixed in the constructor — model works on any graph size
model     = EpidemicTGN(node_feat_dim=9, mem_dim=32, state_dim=5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Cell 10 — Resume from checkpoint if available
ckpts = sorted(glob.glob(f'{BASE}/model/tgn_epoch_*.pt'))
start_epoch = 0
best_val    = float('inf')

if ckpts:
    latest = ckpts[-1]
    ep_num = int(latest.split('tgn_epoch_')[1].replace('.pt',''))
    state  = torch.load(latest, map_location=device)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    best_val    = state.get('best_val', float('inf'))
    start_epoch = ep_num
    print(f'Resumed from epoch {ep_num}, best_val={best_val:.4f}')
else:
    print('Starting from scratch.')

In [ ]:
# Cell 11 — Generalization check helper
# Runs model on an unseen seeding scenario and checks if spread is predicted
@torch.no_grad()
def generalization_check(pair):
    model.eval()
    gseed = pair['graph_seed']
    g     = graphs[gseed]

    # Infect top-degree nodes (may differ from training seeds)
    src_g = g['edge_index'][0]
    deg   = torch.zeros(g['N'], dtype=torch.long, device=device)
    deg.scatter_add_(0, src_g, torch.ones(src_g.shape[0], dtype=torch.long, device=device))
    top3  = torch.topk(deg, k=3).indices

    x0_new = torch.zeros(g['N'], 5, device=device)
    x0_new[:, 0] = 1.0
    for nid in top3:
        x0_new[nid, 0] = 0.0
        x0_new[nid, 2] = 1.0

    pred = model.rollout(
        x0_new, g['node_feats'], g['edge_index'], g['edge_attr'],
        n_days=30, tf_prob=0.0
    )
    I_total = pred[:, :, 2].sum(dim=1).cpu()
    peak_I  = I_total.max().item()
    peak_d  = I_total.argmax().item() + 1

    status = 'OK' if peak_I > 50 else 'WARN: low spread'
    print(f'  GenCheck seed={gseed}: peak_I={peak_I:.0f} on day {peak_d} [{status}]')
    return peak_I

In [ ]:
# Cell 12 — Validation function
@torch.no_grad()
def validate(val_pairs):
    model.eval()
    total = 0.0
    for p in val_pairs:
        gseed  = p['graph_seed']
        g      = graphs[gseed]
        x0     = p['x0'].to(device)
        y      = p['y'].to(device)
        pred   = model.rollout(
            x0, g['node_feats'], g['edge_index'], g['edge_attr'],
            n_days=30, tf_prob=0.0
        )
        loss   = rollout_loss(pred, y, x0, g['edge_index'], g['edge_attr'])
        total += loss.item()
        torch.cuda.empty_cache()
    return total / len(val_pairs)

In [ ]:
# Cell 13 — Training loop
MAX_EPOCHS       = 100
PATIENCE         = 15
patience_counter = 0
train_losses     = []
val_losses       = []

print(f'Training from epoch {start_epoch+1} to {MAX_EPOCHS}')
print(f'Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}')
print('-' * 70)

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    current_tf = tf_prob(epoch)

    random.shuffle(train_pairs)

    for p in train_pairs:
        gseed = p['graph_seed']
        g     = graphs[gseed]
        x0    = p['x0'].to(device)
        y     = p['y'].to(device)

        optimizer.zero_grad(set_to_none=True)

        pred = model.rollout(
            x0, g['node_feats'], g['edge_index'], g['edge_attr'],
            n_days=30, targets=y, tf_prob=current_tf, tbptt=10
        )
        loss = rollout_loss(pred, y, x0, g['edge_index'], g['edge_attr'])
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        del pred, loss
        torch.cuda.empty_cache()

    scheduler.step()
    avg_train = epoch_loss / len(train_pairs)
    train_losses.append(avg_train)

    avg_val = validate(val_pairs)
    val_losses.append(avg_val)

    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch+1:3d} | train={avg_train:.4f} | val={avg_val:.4f} | '
          f'tf={current_tf:.2f} | lr={lr_now:.2e}')

    # Generalization check every 10 epochs
    if (epoch + 1) % 10 == 0 and val_pairs:
        generalization_check(val_pairs[0])

    # Checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch':     epoch + 1,
            'best_val':  best_val,
        }, f'{BASE}/model/tgn_epoch_{epoch+1}.pt')
        print(f'  -> Checkpoint saved epoch {epoch+1}')

    if avg_val < best_val:
        best_val = avg_val
        patience_counter = 0
        torch.save(model.state_dict(), f'{BASE}/model/tgn_best.pt')
        print(f'  -> Best! val={best_val:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'\nDone. Best val loss: {best_val:.4f}')

In [ ]:
# Cell 14 — Loss curve
plt.figure(figsize=(12, 4))
plt.plot(train_losses, label='Train', color='steelblue', linewidth=2)
plt.plot(val_losses,   label='Val',   color='orange',    linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training and validation loss')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/training_curve.png', dpi=150)
plt.show()

In [ ]:
# Cell 15 — Evaluate on TEST graphs (never seen during training)
print('=== TEST SET EVALUATION ===')
print('These graphs were never seen during training.')
print('Good results here = model learned dynamics, not graph identity.')
print()

model.load_state_dict(torch.load(f'{BASE}/model/tgn_best.pt', map_location=device))
model.eval()

all_mae, all_peak_err, all_peak_day_err = [], [], []

for p in test_pairs:
    gseed = p['graph_seed']
    g     = graphs[gseed]
    x0    = p['x0'].to(device)
    y     = p['y'].to(device)

    with torch.no_grad():
        pred = model.rollout(
            x0, g['node_feats'], g['edge_index'], g['edge_attr'],
            n_days=30, tf_prob=0.0
        )

    mae         = (pred[:, :, 2] - y[:, :, 2]).abs().mean().item()
    pred_peak   = pred[:, :, 2].sum(dim=1).max().item()
    true_peak   = y[:, :, 2].sum(dim=1).max().item()
    pred_pday   = pred[:, :, 2].sum(dim=1).argmax().item() + 1
    true_pday   = y[:, :, 2].sum(dim=1).argmax().item()   + 1

    all_mae.append(mae)
    all_peak_err.append(abs(pred_peak - true_peak))
    all_peak_day_err.append(abs(pred_pday - true_pday))

    print(f'  seed={gseed} sim={p["sim_idx"]}: '
          f'MAE={mae:.4f} | '
          f'peak_err={abs(pred_peak-true_peak):.0f} nodes | '
          f'peak_day_err={abs(pred_pday-true_pday)} days')

    torch.cuda.empty_cache()

print()
print(f'Mean MAE on I state:       {np.mean(all_mae):.4f}')
print(f'Mean peak-I count error:   {np.mean(all_peak_err):.1f} nodes')
print(f'Mean peak day error:       {np.mean(all_peak_day_err):.1f} days')

In [ ]:
# Cell 16 — Visual: pred vs truth on one test pair
p     = test_pairs[0]
gseed = p['graph_seed']
g     = graphs[gseed]
x0    = p['x0'].to(device)
y     = p['y'].to(device)

with torch.no_grad():
    pred = model.rollout(
        x0, g['node_feats'], g['edge_index'], g['edge_attr'],
        n_days=30, tf_prob=0.0
    )

# Aggregate to population-level fractions
N_test      = x0.shape[0]
true_totals = (y.cpu().sum(dim=1)   / N_test)   # (30, 5)
pred_totals = (pred.cpu().sum(dim=1) / N_test)  # (30, 5)

days   = list(range(1, 31))
labels = ['S','E','I','R','D']
colors = ['steelblue','orange','red','green','gray']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, totals, title in zip(
    axes,
    [true_totals, pred_totals],
    [f'Ground truth (graph seed={gseed})', 'TGN prediction']
):
    for i, (lbl, col) in enumerate(zip(labels, colors)):
        ax.plot(days, totals[:, i].numpy(), label=lbl, color=col, linewidth=2)
    ax.set_xlabel('Day'); ax.set_ylabel('Population fraction')
    ax.set_title(title)
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f'TGN on unseen test graph (seed={gseed}) — 30 day rollout', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/test_prediction.png', dpi=150)
plt.show()

print(f'True peak I: {true_totals[:,2].max():.3f} on day {true_totals[:,2].argmax().item()+1}')
print(f'Pred peak I: {pred_totals[:,2].max():.3f} on day {pred_totals[:,2].argmax().item()+1}')

In [ ]:
# Cell 17 — CRITICAL: test on a completely new seeding scenario
# This is the real test: give the model seeds it NEVER saw, on a graph it NEVER saw
# If it still predicts spread, the model learned dynamics

print('=== NOVEL SEEDING TEST ===')
print('Seeding with bottom-degree nodes on an unseen graph.')
print('Model has NEVER seen this combination. If it predicts spread, it generalizes.\n')

gseed = 99  # unseen test graph
g     = graphs[gseed]

src_g = g['edge_index'][0]
deg   = torch.zeros(g['N'], dtype=torch.long, device=device)
deg.scatter_add_(0, src_g, torch.ones(src_g.shape[0], dtype=torch.long, device=device))

# Scenario A: seed top-degree (super-spreaders)
top_seeds = torch.topk(deg.float(), k=5).indices
x0_top = torch.zeros(g['N'], 5, device=device)
x0_top[:, 0] = 1.0
for nid in top_seeds: x0_top[nid, 0]=0; x0_top[nid, 2]=1

# Scenario B: seed low-degree nodes
bot_seeds = torch.topk(-deg.float(), k=5).indices
x0_bot = torch.zeros(g['N'], 5, device=device)
x0_bot[:, 0] = 1.0
for nid in bot_seeds: x0_bot[nid, 0]=0; x0_bot[nid, 2]=1

with torch.no_grad():
    pred_top = model.rollout(x0_top, g['node_feats'], g['edge_index'], g['edge_attr'], n_days=30, tf_prob=0.0)
    pred_bot = model.rollout(x0_bot, g['node_feats'], g['edge_index'], g['edge_attr'], n_days=30, tf_prob=0.0)

I_top = pred_top[:, :, 2].sum(dim=1).cpu()
I_bot = pred_bot[:, :, 2].sum(dim=1).cpu()

plt.figure(figsize=(12, 5))
plt.plot(range(1,31), I_top.numpy(), label='Seeded top-degree (super-spreaders)', color='red',  linewidth=2)
plt.plot(range(1,31), I_bot.numpy(), label='Seeded low-degree nodes',              color='blue', linewidth=2)
plt.xlabel('Day'); plt.ylabel('Total infected nodes')
plt.title(f'Novel seeding test on unseen graph (seed={gseed})')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/novel_seeding_test.png', dpi=150)
plt.show()

print(f'Top-degree seeding peak: {I_top.max():.0f} nodes on day {I_top.argmax().item()+1}')
print(f'Low-degree seeding peak: {I_bot.max():.0f} nodes on day {I_bot.argmax().item()+1}')
print()
if I_top.max() > I_bot.max():
    print('GOOD: super-spreader seeding causes more infections than low-degree seeding.')
    print('      Model has learned that high-connectivity nodes spread more.')
else:
    print('WARNING: model does not distinguish seeding strategy.')
    print('         Consider more training epochs or more graph diversity.')

In [ ]:
# Cell 18 — Save outputs
import shutil
shutil.copy(f'{BASE}/model/tgn_best.pt',       f'{BASE}/tgn_best.pt')
shutil.copy(f'{BASE}/training_curve.png',       f'{BASE}/training_curve.png')
shutil.copy(f'{BASE}/test_prediction.png',      f'{BASE}/test_prediction.png')
shutil.copy(f'{BASE}/novel_seeding_test.png',   f'{BASE}/novel_seeding_test.png')

print('Output files:')
for f in os.listdir(BASE):
    if not os.path.isdir(f'{BASE}/{f}'):
        mb = os.path.getsize(f'{BASE}/{f}') / 1e6
        print(f'  {f}  ({mb:.1f} MB)')

print('\nDownload tgn_best.pt and promote to a Kaggle dataset called epidemic-model.')